# Bounded evidence agent
Can a short tool loop finish with source-backed output?

In [ ]:
import os
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root / 'src'))
import torch
torch.set_num_threads(1)
torch.manual_seed(42)
smoke = os.environ.get('PYTHONIC_SMOKE') == '1'

In [ ]:
from dataclasses import asdict
from pythonic.agent import Agent, SearchReadClassify
from pythonic.biomedical import EvidencePrediction
from pythonic.data import fixtures
from pythonic.retrieval import BM25
from pythonic.tools import EvidenceTools
sections = [section for paper in fixtures() for section in paper.sections]
tools = EvidenceTools(BM25(sections), sections, lambda q, c, b: EvidencePrediction('yes', 0.7, len(c)), len)
agent = Agent(tools, SearchReadClassify(budget=100), max_turns=8)
state = agent.run('Does treatment reduce fever?')
print(state.status)
for call in state.calls:
    print(call.name, call.arguments)
print(state.answer)

Action selection is a deterministic policy; the fixture classifier is fixed. The loop also accepts an external selector, with the same limits and tool validation.